See the shared transformation contract: real dates are retained, source populations stay distinct, and final CSVs are published only after validation.


In [ ]:
import json
import sys
import time
from pathlib import Path

from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data_prep' / 'transform' / 'common.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the project folder or data_prep/transform.')
TRANSFORM_DIR = PROJECT_ROOT / 'data_prep' / 'transform'
sys.path.insert(0, str(TRANSFORM_DIR))


import pandas as pd

import behavior
import retail
from common import SHARED_WINDOW_START, uid, DEMO_STORE_ID
from contract import publish, stage_directory, validate_frames
from validate import validate

def raw_dataset_path(handle, required_files):
    """Use local raw files when present; otherwise download from Kaggle."""
    dataset_dir = PROJECT_ROOT / 'data_prep' / 'raw' / handle.split('/')[-1]
    if all(any((dataset_dir / name).is_file() for name in alternatives)
           for alternatives in required_files):
        print(f'Using local dataset: {dataset_dir}')
        return dataset_dir
    import kagglehub
    print(f'Downloading dataset to: {dataset_dir}')
    return Path(kagglehub.dataset_download(handle, output_dir=str(dataset_dir)))

BEHAVIOR_RAW = raw_dataset_path(
    'mkechinov/ecommerce-behavior-data-from-multi-category-store',
    list(zip(behavior.RAW_FILES, behavior.RAW_FILES_GZ)))
RETAIL_RAW = raw_dataset_path(
    'mashlyn/online-retail-ii-uci',
    [tuple(retail.RAW_FILES + retail.RAW_FILES_XLSX)])
OUT_DIR = PROJECT_ROOT / 'data_prep' / 'processed'
BEHAVIOR_OUT_DIR = OUT_DIR / 'ecommerce-behavior-data-from-multi-category-store'
RETAIL_OUT_DIR = OUT_DIR / 'online-retail-ii-uci'
CHUNK_SIZE = 250_000
STORE_ID = DEMO_STORE_ID

print(f'behavior : {BEHAVIOR_RAW}')
print(f'retail   : {RETAIL_RAW}')
print(f'output   : {OUT_DIR}')
print('timeline : real source dates preserved')
def csv_output_path(dataset_dir, table):
    table_dir = dataset_dir / table
    table_dir.mkdir(parents=True, exist_ok=True)
    return table_dir / f'{table}.csv'


In [ ]:
# Sample whole sessions across both complete monthly files BEFORE preprocessing.
# A fixed seed makes selection repeatable across chunks and months. Approximately
# 40% of rows per file are retained; exact counts vary because sessions differ in size.
# Raw files stay unchanged; no intermediate sampled CSVs are written.
BEHAVIOR_SAMPLE_FRACTION = 0.40
BEHAVIOR_SAMPLE_SEED = 42
behavior_paths = behavior.resolve_raw_files(BEHAVIOR_RAW)
offset = 0  # Preserve source UTC timestamps.


In [ ]:
# Stream events into staging; final publication follows full validation.
behavior_stage = stage_directory(BEHAVIOR_OUT_DIR)
events_path = behavior_stage / 'events' / 'events.csv'

b_result = behavior.TransformResult()
started = time.time()
b_frames = behavior.transform_stream(
    behavior_paths, STORE_ID, b_result, events_path, offset, CHUNK_SIZE,
    sample_fraction=BEHAVIOR_SAMPLE_FRACTION, sample_seed=BEHAVIOR_SAMPLE_SEED,
)
print(f'  {b_result.events_written:,} events written to {events_path.name} '
      f'in {time.time() - started:.1f}s')
for table, frame in b_frames.items():
    print(f'  {table:<20} {len(frame):>10,} rows')

In [ ]:
# Online Retail II: preserve real source dates and line timestamps.
r_raw = retail.load_raw(RETAIL_RAW)
for name, df in r_raw.items():
    print(f'  {name:<32} {len(df):>12,} rows')

r_result = retail.transform(r_raw, STORE_ID)
for table, frame in r_result.frames.items():
    print(f'  {table:<20} {len(frame):>10,} rows')

In [ ]:
# ---- merge the shared dimensions under one tenant -------------------------
# Both datasets are namespaced (evb: / ret:), so ids can never collide inside
# UNIQUE (store_id, source_customer_id) / (store_id, source_product_id).
frames = {
    'stores': (
        pd.concat([b_frames['stores'], r_result.frames['stores']], ignore_index=True)
        .drop_duplicates(subset='id')
    ),
    'customers': (
        pd.concat([b_frames['customers'], r_result.frames['customers']], ignore_index=True)
        .drop_duplicates(subset=['store_id', 'source_customer_id'])
    ),
    'products': (
        pd.concat([b_frames['products'], r_result.frames['products']], ignore_index=True)
        .drop_duplicates(subset=['store_id', 'source_product_id'])
    ),
    'orders': r_result.frames['orders'],
    'order_items': r_result.frames['order_items'],
}
print({t: len(f) for t, f in frames.items()})

In [ ]:
failures, exceptions = validate_frames(frames)
print('Dimension validation:', failures or 'PASS')
# Every streamed event is checked during publication, regardless of file size.


In [ ]:
behavior_manifest = publish(
    BEHAVIOR_OUT_DIR, b_frames, behavior_paths, b_result.raw_counts,
    dropped=b_result.dropped, notes=b_result.notes, stage=behavior_stage,
    streams={'events': events_path},
    expected_counts={'events': sum(b_result.sampled_counts.values()) - sum(b_result.dropped_by_source.values())},
    accepted_source_exclusions={k: v for k, v in b_result.dropped_by_source.items() if v},
    retain_previous=False,
)
retail_manifest = publish(
    RETAIL_OUT_DIR, r_result.frames, [retail.resolve_raw_file(RETAIL_RAW)], r_result.raw_counts,
    dropped=r_result.dropped, notes=r_result.notes, source_exceptions=r_result.exceptions,
    expected_counts={'order_items': sum(r_result.raw_counts.values()) - sum(r_result.dropped_by_source.values())},
    accepted_source_exclusions={k: v for k, v in r_result.dropped_by_source.items() if v},
    retain_previous=False,
)
print('Both datasets validated and published.')
